# LAMAR-m6A GEO 训练数据处理 Notebook

这个 notebook 是 GitHub 友好版本，用来从 GEO BED、GRCh37 GTF 和 GRCh37 FASTA 复现五个细胞系的训练数据。

主要原则：

1. 文件名尽量保持和你旧代码示例一致。
2. 文件夹名改成更适合 GitHub 的结构。
3. 每一步都有明确输入和输出。
4. 默认不额外生成旧参考中没有的中间文件。


## Step 0. 参数设置

输出目录放在 GitHub 项目的 `data/` 目录中：

```text
data/
  intermediate/full_index/
  processed/atlas/
  processed/full_sequences/
  processed/full_negative_pools/
  processed/full_balanced/
  processed/full_lengths/
  processed/full_splits/
  processed/full_tokenized/
```

你需要先修改 `CONFIG_PATH`、`OUT_DIR`、`M6A_ATLAS_TABLE`、`TOKENIZER_PATH` 和可选的 `ATLAS_SEQ_CSV`。

`M6A_ATLAS_TABLE` 对应 `m6aatlas-seq-new.ipynb` 使用的原始文件 `hg38_CL_Tech.txt`。


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/picb/rnasys2/lidaoqian/LAMAR-M6A


In [4]:

CONFIG_PATH = Path("geo_training_config.example.json")
OUT_DIR = Path("data")

CELL_LINES = ["a549", "cd8t", "hek293", "hela", "molm13"]
LENGTHS = [1601, 801, 401, 201, 101, 51]
POOL_NEGATIVE_RATIO = 100
FINAL_NEGATIVE_RATIO = 10
RANDOM_SEED = 42

# m6A-Atlas 原始表，对应 m6aatlas-seq-new.ipynb 的 hg38_CL_Tech.txt。
M6A_ATLAS_TABLE = Path("data/raw/m6a_atlas/hg38_CL_Tech.txt")
USE_ATLAS_FILTER = True

# 可选：如果你已经有额外的 Atlas seq CSV，也可以填这里；否则保持 None。
ATLAS_SEQ_CSV = None

# 可选：如果需要生成 HuggingFace Dataset，填你的 tokenizer 路径；否则保持 None。
TOKENIZER_PATH = None

DIRS = {
    "index": OUT_DIR / "intermediate" / "full_index",
    "atlas": OUT_DIR / "processed" / "atlas",
    "full_sequences": OUT_DIR / "processed" / "full_sequences",
    "negative_pools": OUT_DIR / "processed" / "full_negative_pools",
    "balanced": OUT_DIR / "processed" / "full_balanced",
    "lengths": OUT_DIR / "processed" / "full_lengths",
    "splits": OUT_DIR / "processed" / "full_splits",
    "tokenized": OUT_DIR / "processed" / "full_tokenized",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

DIRS


{'index': PosixPath('data/intermediate/full_index'),
 'atlas': PosixPath('data/processed/atlas'),
 'full_sequences': PosixPath('data/processed/full_sequences'),
 'negative_pools': PosixPath('data/processed/full_negative_pools'),
 'balanced': PosixPath('data/processed/full_balanced'),
 'lengths': PosixPath('data/processed/full_lengths'),
 'splits': PosixPath('data/processed/full_splits'),
 'tokenized': PosixPath('data/processed/full_tokenized')}

## Step 1. 导入函数并读取配置

对应旧代码：这是把旧 notebook 中散落的 import、路径变量、辅助函数整理成统一入口。

输入：

```text
geo_training_config.example.json
prepare_geo_training_data.py
```

输出：无文件输出。


In [5]:
import json
import pickle
import pandas as pd
from pathlib import Path

from lamar_m6a.data.prepare_geo_training_data import (
    read_gtf_exons,
    read_fasta,
    build_full_transcripts,
    build_chrom_index,
    map_bed_to_transcript_sites,
    motif_a_positions,
    pad_and_extract,
    split_by_label,
    make_center_101_index,
)

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
config


/opt/anaconda3/envs/torch201/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


FileNotFoundError: [Errno 2] No such file or directory: 'geo_training_config.example.json'

## Step 2. 读取 GTF/FASTA 并构建 transcript full sequence

对应 `newdesign-origin.ipynb`：

- 读取基因组注释文件
- 读取基因组序列文件
- 提取 exon/CDS/transcript 信息
- 构建转录本全长序列

输入来自配置文件：

```text
data/raw/reference/Homo_sapiens.GRCh37.87.gtf.gz
data/raw/reference/Homo_sapiens.GRCh37.dna.primary_assembly.fa.gz
```

输出：无文件输出，只生成内存变量 `exons / genome / transcripts / tx_by_key / by_chrom`。


In [ ]:
ref = config["reference"]

exons = read_gtf_exons(ref["gtf"], ref.get("transcript_biotype"))
genome = read_fasta(ref["fasta"])
transcripts = build_full_transcripts(exons, genome)

tx_by_key = {(tx.gene_id, tx.transcript_id): tx for tx in transcripts}
by_chrom = build_chrom_index(transcripts)

print("exons:", exons.shape)
print("chromosomes:", len(genome))
print("transcripts:", len(transcripts))


## Step 3. GEO BED -> 1601nt positive/negative full sequence CSV

对应 `newdesign-origin.ipynb` 的 GEO 位点处理和正负样本导出逻辑。

输入：配置文件中五个细胞系的 BED 文件。

输出：

```text
data/processed/full_sequences/m6a_positive_{cell}.csv
data/processed/full_sequences/m6a_negative_{cell}.csv
```

说明：positive 是实验确认的 m6A A 位点；negative 是同一批 transcript 中其他符合 motif 的 A 位点，并排除所有细胞系中的 positive 位点。


In [ ]:
positive_by_cell = {}
global_positive = {}

for cell, meta in config["cell_lines"].items():
    if cell not in CELL_LINES:
        continue
    pos_df = map_bed_to_transcript_sites(meta["bed_files"], by_chrom)
    pos_df["cell_line"] = cell
    positive_by_cell[cell] = pos_df
    for row in pos_df.itertuples():
        key = (row.gene_id, row.transcript_id)
        global_positive.setdefault(key, set()).add(int(row.transcript_pos_0based))
    print(cell, "positive sites:", len(pos_df))

for cell, pos_df in positive_by_cell.items():
    pos_records = []
    neg_records = []

    for (gene_id, transcript_id), sub in pos_df.groupby(["gene_id", "transcript_id"]):
        tx = tx_by_key.get((gene_id, transcript_id))
        if tx is None:
            continue

        pos_set = set(map(int, sub["transcript_pos_0based"]))
        all_global_pos = global_positive.get((gene_id, transcript_id), set())
        neg_candidates = motif_a_positions(tx.seq) - all_global_pos

        for center in sorted(pos_set):
            pos_records.append({"seq": tx.seq, "center": center, "label": 1, "gene_id": gene_id, "transcript_id": transcript_id})
        for center in sorted(neg_candidates):
            neg_records.append({"seq": tx.seq, "center": center, "label": 0, "gene_id": gene_id, "transcript_id": transcript_id})

    pos_table = pd.DataFrame(pos_records).drop_duplicates(subset=["transcript_id", "center"])
    neg_table = pd.DataFrame(neg_records).drop_duplicates(subset=["transcript_id", "center"])

    pos_table.to_csv(DIRS["full_sequences"] / f"m6a_positive_{cell}.csv", index=False)
    neg_table.to_csv(DIRS["full_sequences"] / f"m6a_negative_{cell}.csv", index=False)

    print(cell, "positive:", len(pos_table), "negative candidates:", len(neg_table))


## Step 4. 构建 1:100 negative 候选池

对应 `dataprocess.ipynb` 前面生成 `m6a_*_sample_100negative.csv` 的逻辑。

输入：

```text
data/processed/full_sequences/m6a_positive_{cell}.csv
data/processed/full_sequences/m6a_negative_{cell}.csv
```

输出：

```text
data/processed/full_negative_pools/m6a_{cell}_sample_100negative.csv
```

这个文件包含所有 positive 加上 100 倍数量的 sampled negative。


In [ ]:
for cell in CELL_LINES:
    pos = pd.read_csv(DIRS["full_sequences"] / f"m6a_positive_{cell}.csv")
    neg = pd.read_csv(DIRS["full_sequences"] / f"m6a_negative_{cell}.csv")

    n_neg = len(pos) * POOL_NEGATIVE_RATIO
    if len(neg) < n_neg:
        raise ValueError(f"{cell}: negative 不足，需要 {n_neg}，实际 {len(neg)}")

    neg_pool = neg.sample(n=n_neg, random_state=RANDOM_SEED).reset_index(drop=True)
    pool = pd.concat([pos, neg_pool], ignore_index=True)
    pool.to_csv(DIRS["negative_pools"] / f"m6a_{cell}_sample_100negative.csv", index=False)

    print(cell, pool["label"].value_counts().to_dict())


## Step 5. 建立中心 101nt 索引

对应 `dataprocess.ipynb` 中 `m6a_full_*_sample100_101_index.pkl` 的逻辑。

输入：

```text
data/processed/full_negative_pools/m6a_{cell}_sample_100negative.csv
```

输出：

```text
data/intermediate/full_index/m6a_full_{cell}_sample100_101_index.pkl
```

这个索引用于后续按中心 101nt 序列快速匹配可能的外部 m6A-Atlas positive，从 negative 池里删除疑似假阴性。


In [ ]:
for cell in CELL_LINES:
    pool = pd.read_csv(DIRS["negative_pools"] / f"m6a_{cell}_sample_100negative.csv")
    index_101 = make_center_101_index(pool)

    out_path = DIRS["index"] / f"m6a_full_{cell}_sample100_101_index.pkl"
    with open(out_path, "wb") as handle:
        pickle.dump(index_101, handle)

    print(cell, "index keys:", len(index_101), "->", out_path)


## Step 6. 从 m6A-Atlas 原始表提取过滤用 seq

对应 `m6aatlas-seq-new.ipynb`，这一部分的目标不是重新做坐标映射，而是从 m6A-Atlas 原始表 `hg38_CL_Tech.txt` 中提取 `SeqCont` 序列，生成后续过滤 negative 要用的 Atlas seq 文件。

输入：

```text
data/raw/m6a_atlas/hg38_CL_Tech.txt
```

输出对齐你已有的 Atlas 结果文件：

```text
data/processed/atlas/cell4_seq.csv
data/processed/atlas/cell5_seq.csv
data/processed/atlas/GSE63753_seq.csv
data/processed/atlas/GSE71154_seq.csv
data/processed/atlas/GSE86336_seq.csv
data/processed/atlas/GSE98623_seq.csv
data/processed/atlas/m6a_atlas_seq_data.csv
```

说明：

- `cell4_seq.csv`：`GSE63753 / GSE71154 / GSE86336`，对应旧代码中先处理的四个细胞系相关 GEO。
- `cell5_seq.csv`：`GSE63753 / GSE71154 / GSE86336 / GSE98623`，对应五个细胞系完整集合。
- `GSE*_seq.csv`：每个 GSE 单独导出的 seq。
- `m6a_atlas_seq_data.csv`：m6A-Atlas 中全部 `SeqCont` 去重序列。

后面的 Step 7 默认使用 `cell5_seq.csv` 来过滤训练 negative。


In [ ]:
ATLAS_COLUMNS = [
    "seqnames", "start", "end", "width", "strand",
    "GSE_tech_species_cellLine_treatment_C_count_Coverage", "SeqCont",
    "annotation", "Gene_Name", "Gene_Source", "Gene_Biotype",
    "Ensembl_ID", "ID", "RBP_Num", "miRNA_Num", "SplicingSite_Num", "SNP_Num",
    "Cell_Line", "Cell_Line_Num", "Technique", "Technique_Num",
]

CELL4_ATLAS_GSES = ["GSE63753", "GSE71154", "GSE86336"]
CELL5_ATLAS_GSES = ["GSE63753", "GSE71154", "GSE86336", "GSE98623"]


def read_m6a_atlas_table(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)
    if "SeqCont" not in df.columns:
        df = pd.read_csv(path, sep="\t", names=ATLAS_COLUMNS, header=0, low_memory=False)
    return df


def extract_gse_seq(atlas_df, target_gses):
    gse_col = "GSE_tech_species_cellLine_treatment_C_count_Coverage"
    pattern = "|".join(target_gses)
    rows = atlas_df[atlas_df[gse_col].astype(str).str.contains(pattern, na=False)]

    extracted = []
    for _, row in rows.iterrows():
        gse_info = str(row[gse_col])
        seq = row["SeqCont"]
        for gse in target_gses:
            if gse in gse_info:
                extracted.append([gse, seq])

    return pd.DataFrame(extracted, columns=["gse", "seq"]).dropna().drop_duplicates().reset_index(drop=True)


atlas_filter_df = pd.DataFrame(columns=["gse", "seq"])

if not USE_ATLAS_FILTER:
    print("USE_ATLAS_FILTER is False, skip m6A-Atlas seq extraction.")
else:
    if not M6A_ATLAS_TABLE.exists():
        raise FileNotFoundError(f"找不到 m6A-Atlas 原始表: {M6A_ATLAS_TABLE}")

    atlas_df = read_m6a_atlas_table(M6A_ATLAS_TABLE)

    # 对应 m6aatlas-seq-new.ipynb 中的 cell4_seq.csv。
    cell4_seq = extract_gse_seq(atlas_df, CELL4_ATLAS_GSES)
    cell4_seq.to_csv(DIRS["atlas"] / "cell4_seq.csv", index=False)

    # 五个细胞系完整集合：在 cell4 的基础上加入 MOLM13/GSE98623。
    cell5_seq = extract_gse_seq(atlas_df, CELL5_ATLAS_GSES)
    cell5_seq.to_csv(DIRS["atlas"] / "cell5_seq.csv", index=False)

    # 每个 GSE 单独保存，文件名对齐你已有目录。
    for gse in CELL5_ATLAS_GSES:
        gse_seq = cell5_seq[cell5_seq["gse"] == gse].drop_duplicates().reset_index(drop=True)
        gse_seq.to_csv(DIRS["atlas"] / f"{gse}_seq.csv", index=False)
        print(f"{gse}_seq.csv:", len(gse_seq))

    # 对应 m6aatlas-seq-new.ipynb 中提取所有 SeqCont。
    all_seq = atlas_df[["SeqCont"]].dropna().drop_duplicates().rename(columns={"SeqCont": "seq"})
    all_seq.to_csv(DIRS["atlas"] / "m6a_atlas_seq_data.csv", index=False)

    # Step 7 默认使用 m6a_atlas_seq_data.csv 过滤训练 negative。
    atlas_filter_df = all_seq.copy()

    print("cell4_seq.csv:", len(cell4_seq))
    print("cell5_seq.csv:", len(cell5_seq))
    print("m6a_atlas_seq_data.csv:", len(all_seq))
    print("Atlas seq used for negative filtering from m6a_atlas_seq_data.csv:", len(atlas_filter_df))


## Step 7. 用 m6A-Atlas 过滤疑似假阴性 negative

对应 `dataprocess.ipynb` 的 Atlas 过滤单元；Atlas seq 来源已经在 Step 6 中由 `m6aatlas-seq-new.ipynb` 的逻辑生成。

输入：

```text
data/processed/full_negative_pools/m6a_{cell}_sample_100negative.csv
data/intermediate/full_index/m6a_full_{cell}_sample100_101_index.pkl
data/processed/atlas/m6a_atlas_seq_data.csv
ATLAS_SEQ_CSV  # 可选额外输入
```

输出：

```text
data/processed/full_negative_pools/m6a_full_{cell}_sample100_allneg_filtered.csv
```

逻辑：把训练 negative 的中心 101nt 与 `m6a_atlas_seq_data.csv` 中 Atlas seq 的中心 101nt 匹配；匹配到的 negative 可能是假阴性，因此从候选池中删除。


In [ ]:
def center_101_from_seq(seq):
    seq = str(seq)
    if len(seq) >= 101:
        start = (len(seq) - 101) // 2
        return seq[start:start + 101]
    return seq


atlas_sources = []

atlas_seq_path = DIRS["atlas"] / "m6a_atlas_seq_data.csv"
if atlas_seq_path.exists():
    atlas_all_seq = pd.read_csv(atlas_seq_path)
    if "seq" not in atlas_all_seq.columns:
        raise ValueError(f"{atlas_seq_path} 必须包含 seq 列")
    atlas_sources.append(atlas_all_seq[["seq"]])
elif "atlas_filter_df" in globals() and not atlas_filter_df.empty:
    # 方便在同一个 notebook 中刚运行完 Step 6 后直接使用内存变量。
    atlas_sources.append(atlas_filter_df[["seq"]])

if ATLAS_SEQ_CSV is not None:
    extra_atlas = pd.read_csv(ATLAS_SEQ_CSV)
    if "seq" not in extra_atlas.columns:
        raise ValueError("ATLAS_SEQ_CSV 必须包含 seq 列")
    atlas_sources.append(extra_atlas[["seq"]])

if atlas_sources:
    atlas_for_filter = pd.concat(atlas_sources, ignore_index=True).drop_duplicates()
    atlas_101 = set(atlas_for_filter["seq"].astype(str).map(center_101_from_seq))
else:
    atlas_for_filter = pd.DataFrame(columns=["seq"])
    atlas_101 = set()

print("Atlas source file:", atlas_seq_path)
print("Atlas 101nt keys for filtering:", len(atlas_101))

for cell in CELL_LINES:
    pool = pd.read_csv(DIRS["negative_pools"] / f"m6a_{cell}_sample_100negative.csv")
    neg = pool[pool["label"] == 0].copy().reset_index(drop=True)

    before = len(neg)
    if atlas_101:
        neg["center_101"] = [pad_and_extract(seq, int(center), 101) for seq, center in zip(neg["seq"], neg["center"])]
        neg = neg[~neg["center_101"].isin(atlas_101)].drop(columns=["center_101"])
    after = len(neg)

    neg.to_csv(DIRS["negative_pools"] / f"m6a_full_{cell}_sample100_allneg_filtered.csv", index=False)
    print(cell, "filtered negative:", after, "removed:", before - after)


## Step 8. 生成 positive:negative = 1:10 母数据集

对应 `dataprocess.ipynb` 中“获取对应细胞系正负样本1:10的数据”。

输入：

```text
data/processed/full_sequences/m6a_positive_{cell}.csv
data/processed/full_negative_pools/m6a_full_{cell}_sample100_allneg_filtered.csv
```

输出：

```text
data/processed/full_balanced/m6a_full_{cell}_sample_pos_10neg.csv
```

这就是你之前一直追踪的 `m6a_full_hek293_sample_pos_10neg.csv` 这一类文件的通用版本。


In [ ]:
for cell in CELL_LINES:
    pos = pd.read_csv(DIRS["full_sequences"] / f"m6a_positive_{cell}.csv")
    neg = pd.read_csv(DIRS["negative_pools"] / f"m6a_full_{cell}_sample100_allneg_filtered.csv")

    n_neg = len(pos) * FINAL_NEGATIVE_RATIO
    if len(neg) < n_neg:
        raise ValueError(f"{cell}: 过滤后 negative 不足，需要 {n_neg}，实际 {len(neg)}")

    final_neg = neg.sample(n=n_neg, random_state=RANDOM_SEED).reset_index(drop=True)
    mother = pd.concat([pos, final_neg], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    mother.to_csv(DIRS["balanced"] / f"m6a_full_{cell}_sample_pos_10neg.csv", index=False)

    print(cell, mother["label"].value_counts().to_dict())


## Step 9. 从 1601nt 母表截取不同长度

对应 `dataprocess.ipynb` 中“分割不同细胞不同长度”。

输入：

```text
data/processed/full_balanced/m6a_full_{cell}_sample_pos_10neg.csv
```

输出：

```text
data/processed/full_lengths/m6a_full_{cell}_{length}.csv
```

长度包括 `1601, 801, 401, 201, 101, 51`。


In [ ]:
for cell in CELL_LINES:
    mother = pd.read_csv(DIRS["balanced"] / f"m6a_full_{cell}_sample_pos_10neg.csv")
    for length in LENGTHS:
        df = mother.copy()
        df["seq"] = [pad_and_extract(seq, int(center), length) for seq, center in zip(df["seq"], df["center"])]
        df = df[["seq", "label", "gene_id", "transcript_id", "center"]]
        df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        df.to_csv(DIRS["lengths"] / f"m6a_full_{cell}_{length}.csv", index=False)
        print(cell, length, df.shape, sorted(df["seq"].str.len().unique()))


## Step 10. 划分 train / valid / test

对应 `dataprocess.ipynb` 中训练集、验证集、测试集划分。

输入：

```text
data/processed/full_lengths/m6a_full_{cell}_{length}.csv
```

输出：

```text
data/processed/full_splits/train_m6a_full_{cell}_{length}.csv
data/processed/full_splits/valid_m6a_full_{cell}_{length}.csv
data/processed/full_splits/test_m6a_full_{cell}_{length}.csv
```

默认逻辑：先按 label 分层取 50% test；剩余 50% 中再取 20% valid，所以最终约为 train 40%、valid 10%、test 50%。


In [ ]:
for cell in CELL_LINES:
    for length in LENGTHS:
        df = pd.read_csv(DIRS["lengths"] / f"m6a_full_{cell}_{length}.csv")
        train, valid, test = split_by_label(df, RANDOM_SEED)

        train.to_csv(DIRS["splits"] / f"train_m6a_full_{cell}_{length}.csv", index=False)
        valid.to_csv(DIRS["splits"] / f"valid_m6a_full_{cell}_{length}.csv", index=False)
        test.to_csv(DIRS["splits"] / f"test_m6a_full_{cell}_{length}.csv", index=False)

        print(cell, length, "train", train.shape, "valid", valid.shape, "test", test.shape)


## Step 11. 可选：CSV -> HuggingFace Dataset

对应 `dataprocess.ipynb` 中 `AutoTokenizer + load_dataset + save_to_disk` 的最终处理。

输入：

```text
data/processed/full_splits/train_m6a_full_{cell}_{length}.csv
data/processed/full_splits/valid_m6a_full_{cell}_{length}.csv
data/processed/full_splits/test_m6a_full_{cell}_{length}.csv
TOKENIZER_PATH
```

输出：

```text
data/processed/full_tokenized/full-{cell}/m6a_full_{cell}_{length}/
```

如果 `TOKENIZER_PATH = None`，这一单元会自动跳过。


In [ ]:
if TOKENIZER_PATH is None:
    print("TOKENIZER_PATH is None, skip tokenized dataset generation.")
else:
    import multiprocessing as mp
    from datasets import load_dataset
    from transformers import AutoTokenizer

    def process_function(examples, tokenizer, length):
        tokenized = tokenizer(examples["seq"], max_length=length + 2, truncation=True)
        tokenized["labels"] = examples["label"]
        return tokenized

    for cell in CELL_LINES:
        for length in LENGTHS:
            tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, model_max_length=length + 2)
            dataset = load_dataset("csv", data_files={
                "train": str(DIRS["splits"] / f"train_m6a_full_{cell}_{length}.csv"),
                "valid": str(DIRS["splits"] / f"valid_m6a_full_{cell}_{length}.csv"),
                "test": str(DIRS["splits"] / f"test_m6a_full_{cell}_{length}.csv"),
            })
            processed = dataset.map(
                lambda examples: process_function(examples, tokenizer, length),
                batched=True,
                num_proc=mp.cpu_count(),
                remove_columns=dataset["train"].column_names,
            )
            out_path = DIRS["tokenized"] / f"full-{cell}" / f"m6a_full_{cell}_{length}"
            processed.save_to_disk(out_path, num_proc=mp.cpu_count())
            print("saved", out_path)


## Step 12. 检查结果文件

对应旧 notebook 里散落的 `value_counts()`、序列长度检查和文件存在性检查。

输入：前面步骤生成的关键文件。

输出：无新文件，只打印检查结果。


In [ ]:
for cell in CELL_LINES:
    print()
    print("===", cell, "===")
    key_files = [
        DIRS["full_sequences"] / f"m6a_positive_{cell}.csv",
        DIRS["full_sequences"] / f"m6a_negative_{cell}.csv",
        DIRS["negative_pools"] / f"m6a_{cell}_sample_100negative.csv",
        DIRS["negative_pools"] / f"m6a_full_{cell}_sample100_allneg_filtered.csv",
        DIRS["balanced"] / f"m6a_full_{cell}_sample_pos_10neg.csv",
    ]
    for path in key_files:
        print(path, path.exists())

    mother = pd.read_csv(DIRS["balanced"] / f"m6a_full_{cell}_sample_pos_10neg.csv")
    print("mother label counts:", mother["label"].value_counts().to_dict())

    example = pd.read_csv(DIRS["splits"] / f"train_m6a_full_{cell}_51.csv")
    print("train 51 label counts:", example["label"].value_counts().to_dict())
    print("train 51 seq lengths:", sorted(example["seq"].str.len().unique()))